## Data Quality Assessment & Pipeline Code: Loan Approval Applications (Nested JSON to Modeling-Ready Table)

#### 0. Setup & Data Load

In [8]:
import pandas as pd 
import numpy as np
import ast
import json
import re

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
import joblib

df = pd.read_json("../data/raw_credit_applications.json")



#### 1. Raw Data Inspection
Purpose: confirm schema, types, and missingness before transforming anything

In [9]:
print("Raw shape:", df.shape)
display(df.head(3))

Raw shape: (502, 8)


,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN


In [10]:
# Schema and missingness snapshot
display(df.info())
missing_pct_raw = (df.isna().mean().sort_values(ascending=False) * 100).round(1)
display(missing_pct_raw.to_frame("missing_% (raw)"))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   _id                   502 non-null    object
 1   applicant_info        502 non-null    object
 2   financials            502 non-null    object
 3   spending_behavior     502 non-null    object
 4   decision              502 non-null    object
 5   processing_timestamp  62 non-null     object
 6   loan_purpose          50 non-null     object
 7   notes                 2 non-null      object
dtypes: object(8)
memory usage: 31.5+ KB


None

,missing_% (raw)
notes,99.6
loan_purpose,90.0
processing_timestamp,87.6
_id,0.0
spending_behavior,0.0
financials,0.0
applicant_info,0.0
decision,0.0


#### 2. Helper Functions
Reusable utilities for parsing, boolean standardization, and spending feature extraction


In [11]:
# a) SAFE PARSE (handles both dicts and stringified dicts) 
NULL_LIKE = {"", " ", "na", "n/a", "nan", "none", "null", "nil", "undefined"}

def safe_parse(val):
    """
    Parse dict/list if already native, or parse JSON / python-literal strings.
    Returns {} when parsing fails (best for dict-like columns).
    """
    if val is None:
        return {}
    if isinstance(val, (dict, list)):
        return val
    if isinstance(val, float) and pd.isna(val):
        return {}
    if isinstance(val, str):
        s = val.strip()
        if not s or s.lower() in NULL_LIKE:
            return {}
        # Handles true/false/null
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass
        try:
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            return {}
    return {}


In [12]:
# b) TO_BOOL
def to_bool(x):
    if isinstance(x, bool):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, float)):
        return bool(x)
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true", "1", "yes", "y"}:
            return True
        if s in {"false", "0", "no", "n"}:
            return False
    return np.nan


In [13]:
def clean_cat(cat):
    cat = (cat or "unknown").strip().lower()
    cat = re.sub(r"\s+", "_", cat)
    cat = re.sub(r"[^a-z0-9_]", "", cat)
    return cat or "unknown"


#### 3. Flatten Nested Structures (applicant_info, financials, decision)
Normalize nested JSON into a flat tabular structure.
Keep only the label from "decision" to prevent leakage


In [14]:
# Flatten applicant_info 
if "applicant_info" not in df.columns:
    raise ValueError("Column 'applicant_info' not found in dataframe.")

applicant_parsed = df["applicant_info"].apply(safe_parse)
applicant_df = pd.json_normalize(applicant_parsed)
applicant_df.columns = ["applicant_" + c for c in applicant_df.columns]

# Flatten financials 
if "financials" not in df.columns:
    raise ValueError("Column 'financials' not found in dataframe.")

financials_parsed = df["financials"].apply(safe_parse)
financials_df = pd.json_normalize(financials_parsed)
financials_df.columns = ["fin_" + c for c in financials_df.columns]

# Flatten decision (Leakage-safe) 
if "decision" not in df.columns:
    raise ValueError("Column 'decision' not found in dataframe.")

decision_parsed = df["decision"].apply(safe_parse)
decision_df = pd.json_normalize(decision_parsed)
decision_df.columns = ["decision_" + c for c in decision_df.columns]

# Ensure target exists
if "decision_loan_approved" not in decision_df.columns:
    raise ValueError(
        "Column 'decision_loan_approved' not found after flattening decision."
    )

# Keep only the target to prevent leakage
decision_df = decision_df[["decision_loan_approved"]].copy()

print("Flattening complete.")
print("Applicant columns:", applicant_df.columns.tolist())
print("Financial columns:", financials_df.columns.tolist())
print("Decision columns kept:", decision_df.columns.tolist())


Flattening complete.
Applicant columns: ['applicant_full_name', 'applicant_email', 'applicant_ssn', 'applicant_ip_address', 'applicant_gender', 'applicant_date_of_birth', 'applicant_zip_code']
Financial columns: ['fin_annual_income', 'fin_credit_history_months', 'fin_debt_to_income', 'fin_savings_balance', 'fin_annual_salary']
Decision columns kept: ['decision_loan_approved']


#### 4. Spending Behaviour Feature Engineering
Convert transaction lists into structured features (category totals + aggregate behaviour metrics)


In [15]:
def parse_spending(val):
    """
    Convert spending_behavior into:
    - spend_<category> totals
    - aggregate behavior features (txn count, total, mean, max, unique cats)
    """
    # Missing cases
    if val is None:
        return {}
    if isinstance(val, float) and pd.isna(val):
        return {}

    # Ensure list
    items = val if isinstance(val, list) else safe_parse(val)
    if not isinstance(items, list):
        return {}

    out = {}
    amounts = []
    cats_seen = set()

    for item in items:
        if not isinstance(item, dict):
            continue

        cat = clean_cat(item.get("category"))
        amt = item.get("amount", 0)

        try:
            amt = float(amt)
        except (TypeError, ValueError):
            amt = 0.0

        out[f"spend_{cat}"] = out.get(f"spend_{cat}", 0.0) + amt
        amounts.append(amt)
        cats_seen.add(cat)

    # Aggregate spend features (high value)
    out["spend_txn_count"] = int(len(amounts))
    out["spend_total"] = float(sum(amounts))
    out["spend_mean"] = float(np.mean(amounts)) if amounts else 0.0
    out["spend_max"] = float(np.max(amounts)) if amounts else 0.0
    out["spend_unique_cats"] = int(len(cats_seen))

    return out

spending_parsed = df["spending_behavior"].apply(parse_spending)
spending_df = pd.DataFrame(list(spending_parsed)).fillna(0)

print("spending_df shape:", spending_df.shape)
display(spending_df.head(3))



spending_df shape: (502, 20)


,spend_shopping,spend_rent,spend_alcohol,spend_txn_count,spend_total,spend_mean,spend_max,spend_unique_cats,spend_dining,spend_healthcare,spend_fitness,spend_entertainment,spend_insurance,spend_travel,spend_transportation,spend_utilities,spend_groceries,spend_education,spend_adult_entertainment,spend_gambling
0,480.0,790.0,247.0,3,1517.0,505.666667,790.0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,608.0,0.0,3,947.0,315.666667,608.0,3,96.0,243.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,109.0,0.0,1,109.0,109.000000,109.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### 5. Combine into a Modelling-Ready Clean Dataset
Join all extracted feature blocks into a single modelling-ready dataframe


In [16]:
# Combine Everything

base_cols = [c for c in ["_id", "processing_timestamp", "loan_purpose", "notes"] if c in df.columns]

df_clean = pd.concat(
    [
        df[base_cols].reset_index(drop=True),
        applicant_df.reset_index(drop=True),
        financials_df.reset_index(drop=True),
        decision_df.reset_index(drop=True),
        spending_df.reset_index(drop=True),
    ],
    axis=1,
)
